initializing the DistilBert 

In [1]:
import torch
from transformers import DistilBertTokenizer, DistilBertModel

# 1. Check if your GPU is actually awake
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} (If this says CPU, we have a problem)")

# 2. Download/Load the Tokenizer and the Model
# 'distilbert-base-uncased' means it makes all text lowercase (easier for it to read)
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')

# 3. Move the model to your RTX 4050
model = model.to(device)

# 4. We are NOT training, so tell PyTorch to freeze the weights
model.eval() 

print("DistilBERT is locked and loaded on the GPU.")

c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda (If this says CPU, we have a problem)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3852.51it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBERT is locked and loaded on the GPU.


testing DistillBert on a small 100 data points... 

In [2]:
import numpy as np
from tqdm import tqdm
import torch
import pandas as pd

df = pd.read_csv(r"../data/train.csv")
# 1. Grab the text (Replace 'df' with whatever your dataframe is named)
# FILLNA IS CRUCIAL because if BERT sees a NaN (blank space), it will crash instantly.
# Let's just do the first 100 rows to test it so you don't cry. Change this later!
texts = df['FullDescription'].fillna("").head(100).tolist() 

batch_size = 16 # Keep this low (16 or 32). Your 6GB VRAM is weak.
all_embeddings = []

print("Starting the extraction... pray to the GPU gods.")

# 2. The Loop
for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i+batch_size]
    
    # Tokenize: Cuts sentences into pieces. 
    # max_length=128 ensures we don't read huge essays and crash the VRAM.
    inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    
    # Move the chopped up text to your GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Run it through the model WITHOUT gradients (saves massive memory)
    with torch.no_grad():
        outputs = model(**inputs)
        
    # Grab the [CLS] token (the first token of every sentence). 
    # This single token contains the "summary/vibe" of the whole job description.
    cls_embeddings = outputs.last_hidden_state[:, 0, :]
    
    # Move it back to the CPU and convert to numbers so we can save it
    all_embeddings.append(cls_embeddings.cpu().numpy())

# 3. Stack all the batches together into one giant table of numbers
final_features = np.vstack(all_embeddings)

print(f"Done! We got an embedding matrix of shape: {final_features.shape}")

Starting the extraction... pray to the GPU gods.


100%|██████████| 7/7 [00:00<00:00,  9.25it/s]

Done! We got an embedding matrix of shape: (100, 768)


it is working.. so chill, lets do for everything

In [3]:
import pandas as pd
df_train = pd.read_csv(r"../data/train.csv")
df_valid = pd.read_csv(r"../data/valid.csv")
df_test = pd.read_csv(r"../data/test.csv")

In [4]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import DistilBertTokenizer, DistilBertModel

# 1. Point this to YOUR fine-tuned folder!
model_path = '../data/my_finetuned_distilbert'

print("Waking up the Finance Bro BERT...")
tokenizer = DistilBertTokenizer.from_pretrained(model_path)
# We use DistilBertModel (not Classification) because we only want the embeddings!
model = DistilBertModel.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval() # Freeze it!

# 2. Load the RAW data (because we need the text, as you smartly pointed out)
print("Loading raw data...")
df_train = pd.read_csv("../data/train.csv")
df_valid = pd.read_csv("../data/valid.csv")
df_test = pd.read_csv("../data/test.csv")

# Glue them
for df in [df_train, df_valid, df_test]:
    df['Combined_Text'] = df['Title'].fillna("Unknown") + " | " + df['FullDescription'].fillna("Unknown")

# 3. The Extraction Function
def extract_money_vibes(text_series, batch_size=32):
    texts = text_series.tolist()
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        cls_vibe = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_vibe)
        
    return np.vstack(all_embeddings)

# 4. Extract and Overwrite
print("Nuking Train Data with new vibes...")
np.save("../data/train_embeds.npy", extract_money_vibes(df_train['Combined_Text']))

print("Nuking Valid Data with new vibes...")
np.save("../data/valid_embeds.npy", extract_money_vibes(df_valid['Combined_Text']))

print("Nuking Test Data with new vibes...")
np.save("../data/test_embeds.npy", extract_money_vibes(df_test['Combined_Text']))

print("Extraction complete. Absolute cinema.")

Waking up the Finance Bro BERT...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3553.77it/s]
DistilBertModel LOAD REPORT from: ../data/my_finetuned_distilbert
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.weight     | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading raw data...
Nuking Train Data with new vibes...


100%|██████████| 7649/7649 [10:57<00:00, 11.63it/s]


Nuking Valid Data with new vibes...


100%|██████████| 1914/1914 [02:45<00:00, 11.55it/s]


Nuking Test Data with new vibes...


100%|██████████| 1914/1914 [02:45<00:00, 11.58it/s]


Extraction complete. Absolute cinema.
